# Transfer labeling with scANVI

In [1]:
from pathlib import Path
import os
import scanpy as sc
from scipy.sparse import issparse
import scvi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import anndata as ad
from lightning.pytorch import seed_everything
import random
import torch
import sys
import session_info

/home/workspace/environment/sc-charter/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
random.seed(0)
seed_everything(0)

scvi.settings.seed = 0
scvi.settings.num_workers = 32

Seed set to 0
Seed set to 0


### Set paths

In [3]:
base_dir = Path('/home/workspace/projects/madv2')

adata_dir = base_dir / 'data/h5ad/03_scvi'
ref_data_dir = base_dir / 'data/h5ad/04_scanvi' # add the uninfected sample from Reina-Campos et al., 2025 to here

scvi_dir = base_dir / 'scvi'
scanvi_dir = base_dir / 'scanvi' # csv output
output_dir = base_dir / 'data/h5ad/04_scanvi' # h5ad output

scanvi_dir.mkdir(parents=True, exist_ok=True)
ref_data_dir.mkdir(parents= True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

### Read data

In [4]:
adata = sc.read_h5ad(os.path.join(adata_dir, 'madv2-scvi-leiden.h5ad'))
refdata = sc.read_h5ad(os.path.join(ref_data_dir, 'uninfected.h5ad')) # changed from adata/uninfected

/home/workspace/environment/sc-charter/lib/python3.12/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [5]:
refdata.obs_names_make_unique()

In [6]:
refdata.obs['batch'].value_counts()

batch
segmentation_SI1    242156
segmentation_SI2    191024
Name: count, dtype: int64

### Prepare adatas

In [7]:
print(refdata.layers)
print(adata.layers)

Layers with keys: raw
Layers with keys: counts, log1p, normalized_1e6


In [8]:
# restore raw counts, assign same name as query adata
refdata.layers['counts'] = refdata.layers['raw'].copy()  # Store raw counts
refdata.X = refdata.layers['counts'].copy()  # Set X to use raw counts

In [9]:
adata.X = adata.layers['counts'].copy()

In [10]:
adata.X[:5, :5].toarray()  # Convert and preview first 5x5 matrix elements

array([[0., 0., 1., 1., 0.],
       [0., 0., 1., 0., 0.],
       [0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]], dtype=float32)

In [11]:
print(refdata.X[:5, :5])

[[5 0 0 1 0]
 [5 0 0 0 0]
 [4 0 0 0 1]
 [0 0 0 0 0]
 [0 0 0 0 1]]


In [12]:
# add sample_id for refdata
refdata.obs['sample_id'] = refdata.obs['batch']
refdata.obs.sample_id.value_counts()

sample_id
segmentation_SI1    242156
segmentation_SI2    191024
Name: count, dtype: int64

In [13]:
# add column up here for filtering later only on refdata
refdata.obs['ref_data'] = 'Yes'
adata.obs['ref_data'] = 'No'

## Gene alignment and concatenation

In [14]:
# Find the intersection of genes for integration
common_genes = list(set(adata.var_names) & set(refdata.var_names))
len(common_genes)

323

### Create subsets

In [15]:
# Create copies of the datasets, keeping only common genes
adata_subset = adata[:, common_genes].copy()
refdata_subset = refdata[:, common_genes].copy()

In [16]:
print(f"adata before subsetting: {adata.shape[1]}")
print(f"adata after subsetting: {adata_subset.shape[1]}")

print(f"refdata before subsetting: {refdata.shape[1]}")
print(f"refdata after subsetting: {refdata_subset.shape[1]}")

adata before subsetting: 480
adata after subsetting: 323
refdata before subsetting: 480
refdata after subsetting: 323


In [17]:
index_match = (adata_subset.var_names == refdata_subset.var_names).all()
print(f"adata.var_names match: {index_match}")

adata.var_names match: True


# Concatenate

In [18]:
print(adata_subset.obs.columns)
print(refdata_subset.obs.columns)

Index(['cell_id', 'x_centroid', 'y_centroid', 'transcript_counts',
       'control_probe_counts', 'genomic_control_counts',
       'control_codeword_counts', 'unassigned_codeword_counts',
       'deprecated_codeword_counts', 'total_counts', 'cell_area',
       'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample',
       'xenium_run_id', 'sample_id', 'n_genes_by_counts',
       'log1p_n_genes_by_counts', 'log1p_total_counts',
       'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes',
       'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes',
       'doublet_scores', 'predicted_doublets', 'batch', '_scvi_batch',
       '_scvi_labels', 'leiden_scVI_1', 'ref_data'],
      dtype='object')
Index(['total_transcripts', 'nuclear_transcripts', 'cytoplasmic_transcripts',
       'nuclear_transcript_percentage', 'x', 'y', 'topic', 'batch',
       '_scvi_batch', '_scvi_labels', 'celltype_predicted', 'Subtype', 'Type',
       'Immunocentric_Type', 'Class', 'leiden', 'epit

In [19]:
# Concatenate datasets
adata_list = [adata_subset, refdata_subset]

adata_concat = ad.concat(adata_list, 
                  join="outer", # make sure it's outer
                  label = 'scanvi_batch',
                  index_unique="-")

/home/workspace/environment/sc-charter/lib/python3.12/site-packages/anndata/_core/merge.py:1358: UserWarning: Only some AnnData objects have `.raw` attribute, not concatenating `.raw` attributes.
  warn(


In [20]:
adata_concat # confirm number genes

AnnData object with n_obs × n_vars = 2254184 × 323
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'sample', 'xenium_run_id', 'sample_id', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_10_genes', 'pct_counts_in_top_20_genes', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_150_genes', 'doublet_scores', 'predicted_doublets', 'batch', '_scvi_batch', '_scvi_labels', 'leiden_scVI_1', 'ref_data', 'total_transcripts', 'nuclear_transcripts', 'cytoplasmic_transcripts', 'nuclear_transcript_percentage', 'x', 'y', 'topic', 'celltype_predicted', 'Subtype', 'Type', 'Immunocentric_Type', 'Class', 'leiden', 'epithelial_distance', 'crypt_villi_axis', 'epithelial_distance_clipped', 'reference_crypt_villi', 'villi_number'

In [21]:
adata_concat.obs.sample_id.value_counts()

sample_id
Hp_MAdV2_D3         536458
Hp_MAdV2_D12        452605
Hp_alone            452154
MAdV2_alone_D3      379787
segmentation_SI1    242156
segmentation_SI2    191024
Name: count, dtype: int64

### Perform filtering on concatenated adata

In [22]:
sc.pp.calculate_qc_metrics(adata_concat, percent_top=(10, 20, 50, 150), inplace=True) # adds total_counts metric for filtering

In [23]:
print(f"Empty cell check (min, max): {adata_concat.X.sum(axis=1).min()}, {adata_concat.X.sum(axis=1).max()}")

Empty cell check (min, max): 0.0, 1731.0


In [24]:
# or operator ensures that rows satisfying at least one condition are kept
# so if adata.obs.ref_data = No (study samples), then the cell is kept
# OR if adata.obs.ref_data = Yes AND the cell has acceptable total counts the row is also kept

adata_concat = adata_concat[
    (adata_concat.obs["ref_data"] == "No") |  # Keep all non-reference samples
    ((adata_concat.obs["ref_data"] == "Yes") &  # Apply total_counts filter only to reference samples
     (adata_concat.obs["total_counts"] > 20) & 
     (adata_concat.obs["total_counts"] < 800))
].copy()

In [25]:
print(f"Empty cell check (min, max): {adata_concat.X.sum(axis=1).min()}, {adata_concat.X.sum(axis=1).max()}")

Empty cell check (min, max): 6.0, 1731.0


# Examine Batch Effects

In [26]:
adata_concat.X[:5, :5].toarray()

array([[11., 15.,  0.,  0.,  0.],
       [ 2.,  4.,  0.,  0.,  0.],
       [ 3.,  7.,  0.,  0.,  0.],
       [ 1.,  1.,  0.,  0.,  0.],
       [ 2.,  2.,  0.,  0.,  0.]])

In [ ]:
# Normalize and run PCA
sc.pp.normalize_total(adata_concat)
sc.pp.log1p(adata_concat)
sc.pp.pca(adata_concat)

# UMAP colored by sample_id
sc.pp.neighbors(adata_concat)
sc.tl.umap(adata_concat)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot UMAP by sample_id
sc.pl.umap(adata_concat, color="sample_id", show=False, ax=axes[0])
axes[0].set_title("UMAP Colored by Sample (Run)")

# Plot UMAP by ref_data
sc.pl.umap(adata_concat, color="ref_data", show=False, ax=axes[1])
axes[1].set_title("UMAP Colored by Reference Data (Yes/No)")

plt.show()
plt.close()

## Train scVI model

In [ ]:
# Check batch key
adata_concat.obs['sample_id'].value_counts()

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_concat, 
    layer="counts", # use raw count data
    batch_key='ref_data', # biggest batch effect is panel
    categorical_covariate_keys = ['sample_id'] # inter-run variability as a covariate
)

model = scvi.model.SCVI(adata_concat,
                       n_layers = 2,
                       n_latent = 30)
print('starting model training')

model.train(early_stopping=True, 
            enable_progress_bar=True,
            accelerator = 'gpu'
           )

In [ ]:
model.save(os.path.join(scvi_dir, '02_model') , prefix='02_label_transfer_')

In [ ]:
SCVI_LATENT_KEY = "X_scVI_refalign"
adata_concat.obsm[SCVI_LATENT_KEY] = model.get_latent_representation()

### Compute neighbors graph and UMAP

In [ ]:
# neighbors
sc.pp.neighbors(adata_concat, use_rep="X_scVI_refalign", key_added='neighbors_scvi_refalign')
sc.tl.umap(adata_concat, neighbors_key='neighbors_scvi_refalign')

### Export refaligned adata

In [ ]:
# save umap dims to csv
umap_df = pd.DataFrame(adata_concat.obsm['X_umap'], index=adata_concat.obs_names, columns=['umap1', 'umap2'])
filename = os.path.join(scanvi_dir, 'adata_scvi_refalign_umap_coords.csv')
umap_df.to_csv(filename)

In [ ]:
filename = os.path.join(output_dir, 'adata_concat_scvi_refalign.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata_concat.write_h5ad(filename, compression='gzip')

# Checkpoint

In [ ]:
filename = os.path.join(output_dir, 'adata_concat_scvi_refalign.h5ad')
adata_concat = sc.read_h5ad(filename)
adata_concat

### Prep data for scANVI

In [ ]:
adata_concat.obs.columns

In [ ]:
# Add unknown category for original  samples
adata_concat.obs['Subtype'] = adata_concat.obs['Subtype'].cat.add_categories('Unknown')
adata_concat.obs = adata_concat.obs.fillna(value = {'Subtype': 'Unknown'})

In [ ]:
adata_concat.obs['Subtype'].value_counts() # confirm that Subtype is the most granular

Below is important for mapping labels in the unintegrated original adata

In [ ]:
# Subset only query dataset (unlabeled before scANVI training)
query_cells = adata_concat.obs["Subtype"] == "Unknown"
query_cells.to_csv(os.path.join(scanvi_dir, 'query_cells-subtype.csv'))
query_cells

In [ ]:
# fix barcodes; suffix was added during concat
query_cells_copy = query_cells.copy()
query_cells_copy.index = query_cells_copy.index.map(lambda x: x[:-2])
query_cells_copy

## Run scANVI

In [ ]:
# scanvi label transfer
scanvi_model = scvi.model.SCANVI.from_scvi_model(model, 
                                                 adata = adata_concat, 
                                                 unlabeled_category = 'Unknown', # Entries in labels_key to label
                                                 labels_key = 'Subtype') # Column to transfer labels from

scanvi_model.train(max_epochs=20, 
                   accelerator = 'gpu',
                   n_samples_per_label = None) # this is longer training because it's not subsampling

In [ ]:
# save scANVI model
scanvi_model.save(os.path.join(scvi_dir, '03_model_scanvi') , prefix='03_madv2_scanvi_')

### Reference mapping step

In [ ]:
SCANVI_LATENT_KEY = "X_scANVI"
SCANVI_PREDICTION_KEY = "scanvi_labels_xenium"

adata_concat.obsm[SCANVI_LATENT_KEY] = scanvi_model.get_latent_representation(adata_concat)
adata_concat.obs[SCANVI_PREDICTION_KEY] = scanvi_model.predict(adata_concat) # this fills out the labels

In [ ]:
adata_concat.obs['scanvi_labels_xenium'].value_counts()

In [ ]:
filename = os.path.join(output_dir, 'adata_concat-scvi-scanvi-predictions.h5ad')
os.makedirs(os.path.dirname(filename), exist_ok = True)

adata_concat.write_h5ad(filename, compression='gzip')

## Map query labels onto original adata

In [ ]:
# re-read fresh adata with all the genes, this is the query
adata_query = sc.read_h5ad(os.path.join(adata_dir, 'madv2-scvi-leiden.h5ad'))
adata_query.obs.head()

In [ ]:
# create adata_labeled from adata_concat which has the labels
adata_labeled = adata_concat[adata_concat.obs['scanvi_batch'] == '0'].copy() # batch is stored as string

print(adata_labeled.obs['scanvi_batch'].value_counts())
print('')
print(adata_labeled.obs['sample_id'].value_counts())

In [ ]:
# Remove suffix from labeled adata subset from adata_concat
adata_labeled.obs.index = adata_labeled.obs.index.astype(str)
adata_labeled.obs.index = adata_labeled.obs.index.str[:-2]
adata_labeled.obs.head()

In [ ]:
# fix barcodes; suffix was added during concat
#query_cells_copy = query_cells.copy()
#query_cells_copy.index = query_cells_copy.index.map(lambda x: x[:-2])

In [ ]:
query_cells_copy

In [ ]:
query_cells_copy = query_cells_copy[query_cells_copy].index.tolist()
print(set(query_cells_copy).issubset(set(adata_labeled.obs.index)))  # Should return True
print(set(query_cells_copy).issubset(set(adata_query.obs.index)))  # Should return True

In [ ]:
# Transfer predictions back to the original query dataset
adata_query.obs['scanvi_labels_xenium'] = adata_labeled.obs.loc[query_cells_copy, "scanvi_labels_xenium"]
adata_query.obs['scanvi_labels_xenium']

In [ ]:
adata_query.obs["scanvi_labels_xenium"].value_counts()

## Export labeled adata

In [ ]:
h5ad_path = os.path.join(output_dir, 'madv2-scanvi-labels.h5ad')
adata_query.write_h5ad(h5ad_path, compression='gzip')

print(h5ad_path)

## Session info

In [ ]:
print('active IDE: sc-spatial-gpu')
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show()